In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rifat4018/
      /kaggle/input/datasets/rifat4018/sat-ext/
        /kaggle/input/datasets/rifat4018/sat-ext/sat_ext/
          README.md
          audit.py
          config.py
          dataset.py
          infer.py
          metrics.py
          ... +7 more files
      /kaggle/input/datasets/rifat4018/satelite-coco/
        README.dataset.txt
        README.roboflow.txt
        /kaggle/input/datasets/rifat4018/satelite-coco/test/
          1011_jpeg.rf.141ad0f26ab02f1bfee50847febecb85.jpg
          1016_jpeg.rf.0aef8e3664a12b1681da06035362ed54.jpg
          1025_jpeg.rf.ce6b95f0170aab49e8aab8728fca22a8.jpg
          1067_jpeg.rf.b20f17d8c4416f8b8ed687b14a48966a.jpg
          1077_jpeg.rf.0e1e2f1b96f4487629a56a6b93755174.jpg
          1087_jpeg.rf.4324835fa22676557aae5101128ff1ca.jpg
          ... +50 more files
        /kaggle/input/datasets/rifat4018/satelite-coco/train/
          1001_jpeg.rf.056f37938104493a480e839

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rifat4018/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
DATASET = "main"          # preset: which attached export this notebook audits
PREFIX = cfg.DATASETS[DATASET]["prefix"]      # split names: PREFIX + "orig" / PREFIX + "grouped"
USE_ORB = True        # partial-overlap check between tiles; set False only if you are short of time
utils.warn_if_gpu_wasted()
import dataset as ds, audit, split
AUDIT = os.path.join(cfg.AUDIT_OUT, DATASET)
os.makedirs(AUDIT, exist_ok=True)
RAW_ROOT = ds.find_coco_root(DATASET)
print("raw export:", RAW_ROOT)
raw = ds.load_raw(RAW_ROOT)
cmap = ds.category_map(raw)
for s, c in raw.items():
    print(f"{s:<5} categories {[(x['id'], x['name']) for x in c['categories']]}  ->  used as YOLO classes {cmap[s]}")
imgs = ds.image_table(raw)
print("images whose file is missing:", int((~imgs.exists).sum()))

raw export: /kaggle/input/datasets/rifat4018/satelite-coco
train categories [(0, 'objects'), (1, 'plane'), (2, 'ship'), (3, 'vehicle')]  ->  used as YOLO classes {1: 0, 2: 1, 3: 2}
val   categories [(0, 'objects'), (1, 'plane'), (2, 'ship'), (3, 'vehicle')]  ->  used as YOLO classes {1: 0, 2: 1, 3: 2}
test  categories [(0, 'objects'), (1, 'plane'), (2, 'ship'), (3, 'vehicle')]  ->  used as YOLO classes {1: 0, 2: 1, 3: 2}
images whose file is missing: 0


## 1. Counts per split (raw export, before any cleaning)
`copies_per_source` = how many images share one Roboflow source name, e.g. `{3: 435}` means 435 sources with 3 copies each.

In [4]:
counts, cats = audit.split_counts(raw, imgs, cmap)
print(counts.set_index("split").T.to_string())
counts.to_csv(f"{AUDIT}/01_split_counts.csv", index=False)

split                               train              val             test
images                               1305               54               55
missing_files                           0                0                0
unique_source_ids                     435               54               55
copies_per_source                {3: 435}          {1: 54}          {1: 55}
annotations                         44909             2032             1826
non_target_annotations                  0                0                0
empty_images                           30                1                1
neg_x                                 358                0                0
neg_y                                 366                0                0
beyond_right                          360                0                0
beyond_bottom                         293                0                0
w_or_h_le_0                             0                0                0
w_or_h_lt_2p

In [5]:
# reconcile with the numbers printed in the rejected manuscript (paper dataset only)
PAPER = {"train images": 1305, "val images": 54, "test images": 55,
         "train boxes": 44909, "val boxes": 2032, "test boxes": 1826,
         "train negative x": 358, "train negative y": 366}
c = counts.set_index("split")
MEAS = {"train images": c.loc["train", "images"], "val images": c.loc["val", "images"], "test images": c.loc["test", "images"],
        "train boxes": c.loc["train", "annotations"], "val boxes": c.loc["val", "annotations"], "test boxes": c.loc["test", "annotations"],
        "train negative x": c.loc["train", "neg_x"], "train negative y": c.loc["train", "neg_y"]}
rec = pd.DataFrame({"paper": PAPER, "measured": MEAS})
rec["match"] = rec.paper == rec.measured
if DATASET == "main":
    print(rec.to_string())
n_src = imgs.source_id.nunique()
print(f"\nunique source images over all splits: {n_src}  (the paper's '1,414 base images' counts augmented copies)")
rec.to_csv(f"{AUDIT}/01_paper_reconciliation.csv")

                  paper  measured  match
train images       1305      1305   True
val images           54        54   True
test images          55        55   True
train boxes       44909     44909   True
val boxes          2032      2032   True
test boxes         1826      1826   True
train negative x    358       358   True
train negative y    366       366   True

unique source images over all splits: 544  (the paper's '1,414 base images' counts augmented copies)


## 2. Box cleaning (logged)

In [6]:
boxes, clean_log = ds.clean_annotations(raw, cmap)
print(clean_log.groupby(["split", "action"]).size().unstack(fill_value=0).to_string())
clean_log.to_csv(f"{AUDIT}/02_clean_log.csv", index=False)
print("boxes kept:", sum(len(v) for v in boxes.values()))

action  clipped  degenerate_removed  exact_duplicate_removed
split                                                       
test         12                   0                        0
train      1412                  26                      184
val           5                   0                       31
boxes kept: 48526


## 3. Exact and near duplicates (SHA-256, pHash under the 8 flips/rotations)

In [7]:
with utils.timer("hashes"):
    feat, PH = audit.image_features(imgs)
sha_dup = feat[feat.duplicated("sha256", keep=False)].sort_values("sha256")
print("byte-identical images:", len(sha_dup))
ph = audit.phash_pairs(feat, PH)
ph.to_csv(f"{AUDIT}/03_phash_pairs.csv", index=False)
print(f"pHash pairs (distance <= {cfg.PHASH_MAX_HAMMING}): {len(ph)}; across splits: {int(ph.cross_split.sum()) if len(ph) else 0}; "
      f"across DIFFERENT source names: {int((~ph.same_source).sum()) if len(ph) else 0}")
if len(ph):
    print(ph.sort_values("hamming").head(20)[["key_i", "key_j", "hamming", "cross_split", "same_source"]].to_string(index=False))
    cross = ph[ph.cross_split | ~ph.same_source].sort_values("hamming")
    p = audit.pair_montage(feat, cross, f"{AUDIT}/03_phash_pairs_crosssplit.png",
                           "pHash near-duplicates across splits / across source names - CHECK BY EYE")
    print("montage:", p)

[timer] hashes: 5.3 min (321s)
byte-identical images: 0
pHash pairs (distance <= 8): 340; across splits: 1; across DIFFERENT source names: 6
                                                  key_i                                                   key_j  hamming  cross_split  same_source
  train/83_jpeg.rf.2237d7e925e635e004ae39dad2c5d3ce.jpg   train/83_jpeg.rf.90dde5a15d9944b867808a2ed74c3250.jpg        0        False         True
 train/634_jpeg.rf.3b29d56802a2fa119b232dc83dc33364.jpg  train/634_jpeg.rf.b23ab686f659837e061267a33cb47c2e.jpg        0        False         True
 train/518_jpeg.rf.0499c9a4ef547765c292ad0f9d6df319.jpg  train/518_jpeg.rf.4e89d9ee8a80c01f445a6f3836ecc05d.jpg        0        False         True
  train/32_jpeg.rf.02c2ea4061539bf0e1b33788c40f5a25.jpg   train/32_jpeg.rf.97aa7fa1e359791caee311f55794a88e.jpg        0        False         True
  train/36_jpeg.rf.707824f5bc67fb145f2689cc9c41bade.jpg   train/36_jpeg.rf.8a9479c36759ad3470240e590b8e74f7.jpg        0    

## 4. Overlapping tiles (ORB features + RANSAC homography)
One representative per source image (the least-rotated copy). Pairs with at least `ORB_MIN_INLIERS` geometric inliers are
treated as the same ground area. **Look at the montage**: if unrelated scenes appear (repetitive parking lots can fool
the matcher), raise `cfg.ORB_MIN_INLIERS` in this cell and rerun this cell and the ones below.

In [8]:
# cfg.ORB_MIN_INLIERS = 40     # <- raise only if the montage shows unrelated scenes
rep = audit.representatives(feat)
orb = None
if USE_ORB:
    with utils.timer("ORB overlap"):
        orb = audit.orb_pairs(feat.path.values[rep])
    orb["key_i"], orb["key_j"] = feat.key.values[rep[orb.i]], feat.key.values[rep[orb.j]]
    orb["split_i"], orb["split_j"] = feat.src_split.values[rep[orb.i]], feat.src_split.values[rep[orb.j]]
    orb.to_csv(f"{AUDIT}/04_orb_pairs.csv", index=False)
    strong = orb[orb.inliers >= cfg.ORB_MIN_INLIERS]
    print(f"overlapping pairs: {len(strong)} (across splits: {int((strong.split_i != strong.split_j).sum())})")
    print("inlier distribution of the 30 strongest pairs:", orb.inliers.head(30).tolist())
    mont = orb.assign(i=rep[orb.i], j=rep[orb.j])
    audit.pair_montage(feat, mont[mont.inliers >= cfg.ORB_MIN_INLIERS], f"{AUDIT}/04_orb_overlap_top.png",
                       "ORB overlap - strongest pairs", score_col="inliers")
    border = mont[(mont.inliers >= cfg.ORB_MIN_INLIERS * 0.6) & (mont.inliers < cfg.ORB_MIN_INLIERS * 1.6)]
    audit.pair_montage(feat, border, f"{AUDIT}/04_orb_overlap_borderline.png",
                       f"ORB pairs near the threshold ({cfg.ORB_MIN_INLIERS}) - are these the same place?", score_col="inliers")

ORB: 544 representative images, 147,696 pairs, 4 workers
[timer] ORB overlap: 6.7 min (404s)
overlapping pairs: 6 (across splits: 3)
inlier distribution of the 30 strongest pairs: [223, 197, 149, 142, 140, 137, 16, 11, 8, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5]


## 5. Groups and leakage of the original split

In [9]:
feat_g, evidence = audit.build_groups(feat, ph, orb, rep, use_orb=USE_ORB)
leak = audit.leakage_summary(feat_g, ph, orb, rep)
leak["evidence_links"] = evidence
utils.save_json(leak, f"{AUDIT}/05_leakage_summary.json")
feat_g.drop(columns=["path"]).to_csv(f"{AUDIT}/05_images_groups.csv", index=False)
for k, v in leak.items():
    print(f"  {k:<45} {v}")
if leak["largest_group_share_of_sources"] > 0.15:
    print("\nWARNING: one group holds more than 15% of all source images. Check 04_orb_overlap_borderline.png;"
          " if those pairs are NOT the same place, raise cfg.ORB_MIN_INLIERS (cell above) and rerun from there.")

  groups_total                                  532
  groups_spanning_splits                        4
  images_in_spanning_groups                     16
  source_ids_in_more_than_one_split             0
  largest_groups_sources                        [2, 2, 2, 2, 2]
  largest_group_share_of_sources                0.003676470588235294
  phash_pairs_cross_split                       1
  orb_overlap_pairs                             6
  orb_overlap_pairs_cross_split                 3
  val_images_whose_group_has_train_images       3
  test_images_whose_group_has_train_images      1
  evidence_links                                {'source_id': 870, 'phash': 340, 'orb': 6}


## 6. Object sizes (areas in px² on the 640×640 images) and ground-truth samples

In [10]:
sizes = audit.box_sizes(boxes, imgs)
tab = sizes.groupby(["split", "bucket"]).size().unstack(fill_value=0).reindex(columns=[b for b, _, _ in cfg.BUCKETS], fill_value=0)
print(tab.to_string()); print((tab.T / tab.sum(1)).T.round(3).to_string())
tab.to_csv(f"{AUDIT}/06_size_buckets.csv")
print("COCO convention (paper): small = area < 32^2 px^2, medium = 32^2..96^2, large >= 96^2; micro (ours) = area < 16^2")
print(audit.size_figure(sizes, f"{AUDIT}/06_size_distribution.png"))
for s in ("train", "val", "test"):
    print(audit.gt_grid(feat, boxes, f"{AUDIT}/06_gt_samples_{s}.png", s))

bucket  micro  tiny16_32  medium  large
split                                  
test      581        827     384     34
train   13199      17740   12374   1386
val       485        910     564     42
bucket  micro  tiny16_32  medium  large
split                                  
test    0.318      0.453   0.210  0.019
train   0.295      0.397   0.277  0.031
val     0.242      0.455   0.282  0.021
COCO convention (paper): small = area < 32^2 px^2, medium = 32^2..96^2, large >= 96^2; micro (ours) = area < 16^2
/kaggle/working/audit/main/06_size_distribution.png
/kaggle/working/audit/main/06_gt_samples_train.png
/kaggle/working/audit/main/06_gt_samples_val.png
/kaggle/working/audit/main/06_gt_samples_test.png


## 7. Write the two splits

In [11]:
man_o = split.orig_manifest(feat_g)
meta_o = ds.write_split(PREFIX + "orig", man_o, imgs, boxes, log=clean_log, extra=dict(dataset=DATASET, leakage=leak))
man_g, ginfo = split.grouped_manifest(feat_g, boxes)
split.check_no_group_leak(man_g)
meta_g = ds.write_split(PREFIX + "grouped", man_g, imgs, boxes, log=clean_log, extra=dict(dataset=DATASET, leakage=leak, grouping=ginfo))
import shutil; shutil.copytree(AUDIT, os.path.join(cfg.SPLITS_OUT, "audit_" + DATASET), dirs_exist_ok=True)

split 'orig' fingerprint 0291ae6cd605
       images  groups  annotations  empty_images  plane  vehicle  ship
train    1305     428        44699            30   7065    33503  4131
val        54      53         2001             1    259     1613   129
test       55      55         1826             1    188     1611    27
grouped split: 532 groups -> test 80, val 80, train 372 (balance score 0.0152; dropped copies 268)
split 'grouped' fingerprint f7cee44b1365
       images  groups  annotations  empty_images  plane  vehicle  ship
train     984     372        33855            26   5127    25721  3007
val        80      80         2887             2    433     2228   226
test       82      80         2735             0    433     2092   210


'/kaggle/working/splits/audit_main'

## 8. Read this before the GPU notebooks
* `orig` fingerprint and `grouped` fingerprint are printed above. **Every training notebook prints the fingerprint it used;
  they must all be the same** (the aggregation notebook checks it).
* Save Version -> Output -> New Dataset `sat-splits`. Share it with the other accounts (Dataset -> Settings -> Sharing -> add
  the usernames, or make it public if the image licence allows).

In [12]:
print("orig    :", meta_o["fingerprint"], meta_o["summary"])
print("grouped :", meta_g["fingerprint"], meta_g["summary"])
print("files:", sorted(os.listdir(cfg.SPLITS_OUT)))

orig    : 0291ae6cd605 {'train': {'images': 1305, 'groups': 428, 'annotations': 44699, 'empty_images': 30, 'plane': 7065, 'vehicle': 33503, 'ship': 4131}, 'val': {'images': 54, 'groups': 53, 'annotations': 2001, 'empty_images': 1, 'vehicle': 1613, 'ship': 129, 'plane': 259}, 'test': {'images': 55, 'groups': 55, 'annotations': 1826, 'empty_images': 1, 'plane': 188, 'vehicle': 1611, 'ship': 27}}
grouped : f7cee44b1365 {'train': {'images': 984, 'groups': 372, 'annotations': 33855, 'empty_images': 26, 'plane': 5127, 'vehicle': 25721, 'ship': 3007}, 'val': {'images': 80, 'groups': 80, 'annotations': 2887, 'empty_images': 2, 'vehicle': 2228, 'plane': 433, 'ship': 226}, 'test': {'images': 82, 'groups': 80, 'annotations': 2735, 'empty_images': 0, 'vehicle': 2092, 'plane': 433, 'ship': 210}}
files: ['audit_main', 'grouped', 'orig']
